# 📡 Telecom Customer Churn Prediction & Retention Intelligence

**IBM SkillsBuild Data Analytics with AI Academic Internship**  
**Author:** Shivam Singh  
**Dataset:** Telco Customer Churn — https://www.kaggle.com/datasets/blastchar/telco-customer-churn

---

## Project Flow
```
Raw Data → Data Cleaning → EDA → Insights → ML Prediction → Feature Importance → Business Decisions
```

## Table of Contents
1. [Imports & Setup](#1-imports--setup)
2. [Data Loading](#2-data-loading)
3. [Data Cleaning & Preprocessing](#3-data-cleaning--preprocessing)
4. [Exploratory Data Analysis (EDA)](#4-exploratory-data-analysis)
5. [Business KPIs](#5-business-kpis)
6. [Feature Engineering & ML Preprocessing](#6-feature-engineering--ml-preprocessing)
7. [Model Training](#7-model-training)
8. [Model Evaluation](#8-model-evaluation)
9. [Feature Importance & Churn Drivers](#9-feature-importance--churn-drivers)
10. [Churn Probability & Risk Labelling](#10-churn-probability--risk-labelling)
11. [Business Insights](#11-business-insights)
12. [Summary & Conclusions](#12-summary--conclusions)

---
## 1. Imports & Setup

In [ ]:
# Standard libraries
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import joblib

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay
)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
COLORS = {'Churn': '#e74c3c', 'No Churn': '#27ae60'}

print('All libraries imported successfully!')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'sklearn : {__import__("sklearn").__version__}')

---
## 2. Data Loading

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — change DATASET_PATH if your CSV is in a different location
# ---------------------------------------------------------------------------
DATASET_PATH = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

def load_data(path: str) -> pd.DataFrame:
    """Load the Telco Customer Churn CSV dataset."""
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Dataset not found at '{path}'.\n"
            "Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn\n"
            "Place 'WA_Fn-UseC_-Telco-Customer-Churn.csv' in the same folder as this notebook."
        )
    return pd.read_csv(path)

raw_df = load_data(DATASET_PATH)
print(f'Dataset loaded: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns')
raw_df.head()

In [ ]:
# Dataset info
print('=== Column Data Types ===')
print(raw_df.dtypes)
print('\n=== Basic Statistics (Numerical) ===')
raw_df.describe()

In [ ]:
print('=== Missing Values ===')
missing = raw_df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No NaN values in raw CSV.')

# TotalCharges has blank strings — check
blank_total = (raw_df['TotalCharges'].str.strip() == '').sum()
print(f"\nBlank strings in TotalCharges (will become NaN): {blank_total}")

In [ ]:
print('=== Duplicate Rows ===')
print(f'Duplicate rows found: {raw_df.duplicated().sum()}')

print('\n=== Churn Class Distribution ===')
print(raw_df['Churn'].value_counts())
print(f"Churn rate: {raw_df['Churn'].value_counts(normalize=True)['Yes']*100:.2f}%")

---
## 3. Data Cleaning & Preprocessing

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Data cleaning steps:
    1. Strip whitespace from column names and string columns.
    2. Convert TotalCharges to numeric (blank strings → NaN).
    3. Drop rows with NaN TotalCharges (those were new customers with 0 tenure).
    4. Encode Churn column: 'Yes'→1, 'No'→0.
    5. Drop customerID (not a predictor).
    6. Remove duplicate rows.
    7. Ensure SeniorCitizen is integer type.
    """
    df = df.copy()

    # Step 1 — strip whitespace
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()

    # Step 2 — TotalCharges to numeric
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

    # Step 3 — drop NaN TotalCharges
    before = len(df)
    df.dropna(subset=['TotalCharges'], inplace=True)
    print(f'Dropped {before - len(df)} rows with missing TotalCharges (new customers with tenure=0).')

    # Step 4 — encode target
    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

    # Step 5 — drop customerID
    df.drop(columns=['customerID'], inplace=True)

    # Step 6 — remove duplicates
    before = len(df)
    df.drop_duplicates(inplace=True)
    print(f'Removed {before - len(df)} duplicate rows.')

    # Step 7 — SeniorCitizen as int
    df['SeniorCitizen'] = df['SeniorCitizen'].astype(int)

    print(f'\nCleaned dataset: {df.shape[0]} rows × {df.shape[1]} columns')
    return df


df = clean_data(raw_df)
df.head()

In [ ]:
# Verify no missing values remain
print('=== Missing Values After Cleaning ===')
print(df.isnull().sum())
print('\n=== Cleaned Data Types ===')
print(df.dtypes)

In [ ]:
print('=== Summary Statistics After Cleaning ===')
df.describe()

---
## 4. Exploratory Data Analysis

In [ ]:
# Helper function: churn rate by category
def churn_rate_by(df: pd.DataFrame, col: str) -> pd.DataFrame:
    grp = df.groupby(col)['Churn'].agg(['sum', 'count']).reset_index()
    grp.columns = [col, 'Churned', 'Total']
    grp['ChurnRate'] = (grp['Churned'] / grp['Total'] * 100).round(2)
    return grp

print('Helper function defined.')

In [ ]:
# --- 4.1 Overall Churn Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Churn Distribution', fontsize=16, fontweight='bold', y=1.02)

# Count plot
churn_counts = df['Churn'].value_counts()
bars = axes[0].bar(['Retained (0)', 'Churned (1)'], churn_counts.values,
                   color=['#27ae60', '#e74c3c'], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, churn_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[0].set_title('Customer Count by Churn Status', fontsize=13)
axes[0].set_ylabel('Number of Customers')

# Pie chart
labels = [f'Retained\n{churn_counts[0]:,} ({churn_counts[0]/len(df)*100:.1f}%)',
          f'Churned\n{churn_counts[1]:,} ({churn_counts[1]/len(df)*100:.1f}%)']
axes[1].pie(churn_counts.values, labels=labels, colors=['#27ae60', '#e74c3c'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Churn Proportion', fontsize=13)

plt.tight_layout()
plt.show()

print(f"Overall Churn Rate: {df['Churn'].mean()*100:.2f}%")

In [ ]:
# --- 4.2 Churn by Contract Type ---
contract_churn = churn_rate_by(df, 'Contract')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Churn by Contract Type', fontsize=15, fontweight='bold')

# Stacked bar
contract_churn_plot = df.groupby(['Contract', 'Churn']).size().unstack()
contract_churn_plot.columns = ['Retained', 'Churned']
contract_churn_plot.plot(kind='bar', ax=axes[0], color=['#27ae60', '#e74c3c'],
                         edgecolor='white', linewidth=1)
axes[0].set_title('Count by Contract & Churn Status')
axes[0].set_ylabel('Number of Customers')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend()

# Churn rate bar
bars = axes[1].bar(contract_churn['Contract'], contract_churn['ChurnRate'],
                   color=['#e74c3c' if r > 20 else '#f39c12' if r > 10 else '#27ae60'
                          for r in contract_churn['ChurnRate']],
                   edgecolor='white', linewidth=1.5)
for bar, rate in zip(bars, contract_churn['ChurnRate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{rate:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Churn Rate (%) by Contract')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()
print(contract_churn.to_string(index=False))

In [ ]:
# --- 4.3 Churn by Tenure ---
bins = [0, 12, 24, 36, 48, 60, 72]
labels_b = ['0–12', '13–24', '25–36', '37–48', '49–60', '61–72']
df['TenureBin'] = pd.cut(df['tenure'], bins=bins, labels=labels_b, right=True)
tenure_churn = churn_rate_by(df, 'TenureBin')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Churn by Customer Tenure', fontsize=15, fontweight='bold')

axes[0].hist([df[df['Churn']==0]['tenure'], df[df['Churn']==1]['tenure']],
             bins=30, label=['Retained', 'Churned'], color=['#27ae60', '#e74c3c'],
             alpha=0.7, stacked=False)
axes[0].set_title('Tenure Distribution by Churn Status')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].plot(tenure_churn['TenureBin'].astype(str), tenure_churn['ChurnRate'],
             'o-', color='#e74c3c', linewidth=2.5, markersize=8)
for i, row in tenure_churn.iterrows():
    axes[1].text(i, row['ChurnRate'] + 0.5, f"{row['ChurnRate']:.1f}%", ha='center', fontsize=10)
axes[1].set_title('Churn Rate (%) by Tenure Group')
axes[1].set_xlabel('Tenure (months)')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

# Clean up temporary column
df.drop(columns=['TenureBin'], inplace=True)

In [ ]:
# --- 4.4 Churn by Internet Service & Payment Method ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Churn by Internet Service & Payment Method', fontsize=15, fontweight='bold')

for ax, col in zip(axes, ['InternetService', 'PaymentMethod']):
    data = churn_rate_by(df, col)
    bars = ax.bar(data[col], data['ChurnRate'],
                  color=['#e74c3c' if r > 40 else '#f39c12' if r > 20 else '#27ae60'
                         for r in data['ChurnRate']])
    for bar, rate in zip(bars, data['ChurnRate']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{rate:.1f}%', ha='center', fontweight='bold', fontsize=9)
    ax.set_title(f'Churn Rate by {col}')
    ax.set_ylabel('Churn Rate (%)')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

In [ ]:
# --- 4.5 Churn by Support & Security Services ---
service_cols = ['TechSupport', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection']
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Churn by Support & Security Services', fontsize=14, fontweight='bold')

for ax, col in zip(axes, service_cols):
    data = churn_rate_by(df, col)
    sns.barplot(data=data, x=col, y='ChurnRate', ax=ax,
                palette=['#27ae60', '#e74c3c', '#95a5a6'])
    ax.set_title(col, fontsize=11)
    ax.set_ylabel('Churn Rate (%)' if ax == axes[0] else '')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
    for p in ax.patches:
        ax.text(p.get_x() + p.get_width()/2, p.get_height() + 0.3,
                f'{p.get_height():.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# --- 4.6 Churn by Monthly Charges & Total Charges ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Churn by Monthly & Total Charges', fontsize=14, fontweight='bold')

for ax, col in zip(axes, ['MonthlyCharges', 'TotalCharges']):
    ax.hist([df[df['Churn']==0][col], df[df['Churn']==1][col]],
            bins=40, label=['Retained', 'Churned'],
            color=['#27ae60', '#e74c3c'], alpha=0.7)
    ax.set_title(f'{col} Distribution by Churn')
    ax.set_xlabel(col + ' ($)')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()

print('Monthly Charges — Mean by Churn:')
print(df.groupby('Churn')['MonthlyCharges'].mean().rename({0:'Retained', 1:'Churned'}))

In [ ]:
# --- 4.7 Churn by Gender & Senior Citizen ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Churn by Demographics', fontsize=14, fontweight='bold')

for ax, col, title in zip(axes,
                           ['gender', 'SeniorCitizen'],
                           ['Gender', 'Senior Citizen Status']):
    data = churn_rate_by(df, col)
    if col == 'SeniorCitizen':
        data[col] = data[col].map({0: 'Non-Senior', 1: 'Senior'})
    sns.barplot(data=data, x=col, y='ChurnRate', ax=ax,
                palette=['#3b6fd4', '#e74c3c'])
    ax.set_title(f'Churn Rate by {title}')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_xlabel('')
    for p in ax.patches:
        ax.text(p.get_x() + p.get_width()/2, p.get_height() + 0.2,
                f'{p.get_height():.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- 4.8 Correlation Heatmap (numerical features) ---
num_df = df[['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Heatmap — Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 4.9 Churn by Paperless Billing & Partner ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Churn by Billing & Partner Status', fontsize=14, fontweight='bold')

for ax, col in zip(axes, ['PaperlessBilling', 'Partner']):
    data = churn_rate_by(df, col)
    sns.barplot(data=data, x=col, y='ChurnRate', ax=ax,
                palette=['#27ae60', '#e74c3c'])
    ax.set_title(f'Churn Rate by {col}')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_xlabel('')
    for p in ax.patches:
        ax.text(p.get_x() + p.get_width()/2, p.get_height() + 0.2,
                f'{p.get_height():.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 5. Business KPIs

In [ ]:
def calculate_kpis(df: pd.DataFrame) -> dict:
    """Calculate all business KPIs from the cleaned dataset."""
    total   = len(df)
    churned = df['Churn'].sum()
    retained = total - churned
    return {
        'Total Customers'       : total,
        'Churned Customers'     : int(churned),
        'Retained Customers'    : int(retained),
        'Churn Rate (%)'        : round(churned / total * 100, 2),
        'Retention Rate (%)'    : round(retained / total * 100, 2),
        'Avg Monthly Charges'   : round(df['MonthlyCharges'].mean(), 2),
        'Avg Tenure (months)'   : round(df['tenure'].mean(), 2),
        'Total Monthly Revenue' : round(df['MonthlyCharges'].sum(), 2),
    }

kpis = calculate_kpis(df)
print('=== BUSINESS KPIs ===')
for k, v in kpis.items():
    if 'Revenue' in k or 'Charges' in k:
        print(f'  {k:<30}: ${v:,.2f}')
    elif '%' in k:
        print(f'  {k:<30}: {v}%')
    else:
        print(f'  {k:<30}: {v:,}')

---
## 6. Feature Engineering & ML Preprocessing

In [ ]:
# Define feature groups
TARGET_COL = 'Churn'

CATEGORICAL_COLS = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]
NUMERICAL_COLS = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']

def preprocess_data(df: pd.DataFrame):
    """
    Prepare features and target.
    Build ColumnTransformer:
      - StandardScaler for numerical features
      - OneHotEncoder for categorical features
    Returns: X_train, X_test, y_train, y_test, preprocessor, cat_cols, num_cols
    """
    feature_cols = [c for c in df.columns if c != TARGET_COL]
    X = df[feature_cols]
    y = df[TARGET_COL]

    cat_cols = [c for c in CATEGORICAL_COLS if c in X.columns]
    num_cols = [c for c in NUMERICAL_COLS  if c in X.columns]

    preprocessor = ColumnTransformer(transformers=[
        ('num', StandardScaler(),                                   num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    return X_train, X_test, y_train, y_test, preprocessor, cat_cols, num_cols


X_train, X_test, y_train, y_test, preprocessor, cat_cols, num_cols = preprocess_data(df)

print(f'Training set  : {X_train.shape[0]} rows')
print(f'Test set      : {X_test.shape[0]} rows')
print(f'Numerical cols: {num_cols}')
print(f'Categorical cols ({len(cat_cols)}): {cat_cols}')

In [ ]:
# Class distribution in train/test
print('Train set Churn distribution:')
print(y_train.value_counts(normalize=True).rename({0:'Retained', 1:'Churned'}).apply(lambda x: f'{x*100:.2f}%'))
print('\nTest set Churn distribution:')
print(y_test.value_counts(normalize=True).rename({0:'Retained', 1:'Churned'}).apply(lambda x: f'{x*100:.2f}%'))

---
## 7. Model Training

In [ ]:
# Train Logistic Regression
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    ))
])
lr_pipeline.fit(X_train, y_train)
print('Logistic Regression trained.')

# Train Random Forest
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        max_depth=10, random_state=42, n_jobs=-1
    ))
])
rf_pipeline.fit(X_train, y_train)
print('Random Forest trained.')

---
## 8. Model Evaluation

In [ ]:
def evaluate_model(name: str, pipeline, X_test, y_test) -> dict:
    """Evaluate a trained pipeline and return metrics dictionary."""
    y_pred  = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    return {
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_test, y_pred),  4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall'   : round(recall_score(y_test, y_pred),    4),
        'F1 Score' : round(f1_score(y_test, y_pred),        4),
        'ROC-AUC'  : round(roc_auc_score(y_test, y_proba),  4),
        'y_pred'   : y_pred,
        'y_proba'  : y_proba,
    }


lr_metrics = evaluate_model('Logistic Regression', lr_pipeline, X_test, y_test)
rf_metrics = evaluate_model('Random Forest',       rf_pipeline, X_test, y_test)

metrics_df = pd.DataFrame([lr_metrics, rf_metrics]).set_index('Model')
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
print('=== MODEL COMPARISON ===')
print(metrics_df[metric_cols].to_string())

best_name = metrics_df['ROC-AUC'].idxmax()
print(f'\n✅ Best model by ROC-AUC: {best_name} ({metrics_df.loc[best_name, "ROC-AUC"]:.4f})')

In [ ]:
# --- ROC Curves ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Model Evaluation', fontsize=14, fontweight='bold')

# ROC
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
for metrics, color, name in [
        (lr_metrics, '#3b6fd4', 'Logistic Regression'),
        (rf_metrics, '#e74c3c', 'Random Forest')]:
    fpr, tpr, _ = roc_curve(y_test, metrics['y_proba'])
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f"{name} (AUC={metrics['ROC-AUC']:.3f})")
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend()

# Confusion matrix for best model
best_pipe = rf_pipeline if best_name == 'Random Forest' else lr_pipeline
best_metrics = rf_metrics if best_name == 'Random Forest' else lr_metrics
cm = confusion_matrix(y_test, best_metrics['y_pred'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title(f'Confusion Matrix — {best_name}')

plt.tight_layout()
plt.show()

In [ ]:
# --- Classification Reports ---
for name, metrics in [('Logistic Regression', lr_metrics), ('Random Forest', rf_metrics)]:
    print(f'\n=== {name} — Classification Report ===')
    print(classification_report(y_test, metrics['y_pred'], target_names=['No Churn', 'Churn']))

In [ ]:
# --- Visualise metric comparison ---
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metric_cols))
width = 0.35

lr_vals = [lr_metrics[m] for m in metric_cols]
rf_vals = [rf_metrics[m] for m in metric_cols]

ax.bar(x - width/2, lr_vals, width, label='Logistic Regression', color='#3b6fd4', alpha=0.85)
ax.bar(x + width/2, rf_vals, width, label='Random Forest',       color='#e74c3c', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(metric_cols)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.legend()
for rect in ax.patches:
    ax.text(rect.get_x() + rect.get_width()/2, rect.get_height() + 0.01,
            f'{rect.get_height():.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

---
## 9. Feature Importance & Churn Drivers

In [ ]:
def get_feature_importance(pipeline, cat_cols: list, num_cols: list) -> pd.DataFrame:
    """Extract and aggregate feature importances from a trained pipeline."""
    clf = pipeline.named_steps['classifier']
    ohe = pipeline.named_steps['preprocessor'].named_transformers_['cat']
    ohe_cols = list(ohe.get_feature_names_out(cat_cols))
    all_features = num_cols + ohe_cols

    if hasattr(clf, 'feature_importances_'):
        importances = clf.feature_importances_
    else:
        importances = np.abs(clf.coef_[0])

    fi_df = pd.DataFrame({'Feature': all_features, 'Importance': importances})
    fi_df['OriginalFeature'] = fi_df['Feature'].apply(
        lambda x: x.split('_')[0] if any(c in x for c in cat_cols) else x
    )
    fi_agg = (
        fi_df.groupby('OriginalFeature')['Importance'].sum()
        .reset_index().sort_values('Importance', ascending=False)
        .reset_index(drop=True)
    )
    return fi_agg


rf_fi = get_feature_importance(rf_pipeline, cat_cols, num_cols)
lr_fi = get_feature_importance(lr_pipeline, cat_cols, num_cols)

print('=== Top 10 Churn Drivers (Random Forest) ===')
print(rf_fi.head(10).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Top Churn Drivers — Feature Importance', fontsize=14, fontweight='bold')

for ax, fi, title in [
        (axes[0], rf_fi.head(10), 'Random Forest'),
        (axes[1], lr_fi.head(10), 'Logistic Regression (|coef|)')]:
    fi_sorted = fi.sort_values('Importance')
    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(fi_sorted)))
    ax.barh(fi_sorted['OriginalFeature'], fi_sorted['Importance'], color=colors)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Importance Score')
    for i, (val, label) in enumerate(zip(fi_sorted['Importance'], fi_sorted['OriginalFeature'])):
        ax.text(val + 0.0005, i, f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## 10. Churn Probability & Risk Labelling

In [ ]:
# Configurable thresholds
LOW_RISK_THRESHOLD    = 0.30
MEDIUM_RISK_THRESHOLD = 0.60

def generate_predictions(df: pd.DataFrame, pipeline, target_col: str = 'Churn') -> pd.DataFrame:
    """Generate churn probability, predicted class, and risk category for every customer."""
    feature_cols = [c for c in df.columns if c != target_col]
    X = df[feature_cols]
    proba = pipeline.predict_proba(X)[:, 1]
    pred  = (proba >= 0.50).astype(int)
    out = df.copy()
    out['ChurnProbability'] = proba.round(4)
    out['PredictedChurn']   = pred
    out['RiskCategory'] = pd.cut(
        proba,
        bins   = [-0.001, LOW_RISK_THRESHOLD, MEDIUM_RISK_THRESHOLD, 1.001],
        labels = ['Low Risk', 'Medium Risk', 'High Risk']
    )
    return out


pred_df = generate_predictions(df, best_pipe)

print('=== Risk Category Distribution ===')
print(pred_df['RiskCategory'].value_counts())

high_risk = pred_df[pred_df['RiskCategory'] == 'High Risk']
revenue_at_risk = high_risk['MonthlyCharges'].sum()
print(f'\nHigh-Risk Customers  : {len(high_risk):,}')
print(f'Revenue at Risk ($/mo): ${revenue_at_risk:,.0f}')

In [ ]:
# Risk distribution chart
risk_counts = pred_df['RiskCategory'].value_counts().reindex(['Low Risk', 'Medium Risk', 'High Risk'])
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#27ae60', '#f39c12', '#e74c3c']
bars = ax.bar(risk_counts.index, risk_counts.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, risk_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val:,}\n({val/len(pred_df)*100:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Customer Churn Risk Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Customers')
plt.tight_layout()
plt.show()

In [ ]:
# Probability distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(pred_df[pred_df['Churn']==0]['ChurnProbability'], bins=40, alpha=0.7,
        color='#27ae60', label='Actual No Churn')
ax.hist(pred_df[pred_df['Churn']==1]['ChurnProbability'], bins=40, alpha=0.7,
        color='#e74c3c', label='Actual Churn')
ax.axvline(LOW_RISK_THRESHOLD,    color='orange', linestyle='--', lw=2, label=f'Low/Medium ({LOW_RISK_THRESHOLD})')
ax.axvline(MEDIUM_RISK_THRESHOLD, color='red',    linestyle='--', lw=2, label=f'Medium/High ({MEDIUM_RISK_THRESHOLD})')
ax.set_title('Churn Probability Distribution by Actual Churn Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Churn Probability')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

---
## 11. Business Insights

In [ ]:
def generate_business_insights(df, kpis, fi_df, pred_df, model_name) -> dict:
    """
    Auto-generate executive insights from the cleaned dataset.
    - All numerical values are computed at runtime.
    - fi_df must be the feature importance of the BEST model (not always RF).
    - model_name is passed explicitly so the text is accurate.
    - Actions are framed as testable hypotheses, not proven interventions.
    """
    # Contract
    contract_churn  = churn_rate_by(df, 'Contract')
    h_contract      = contract_churn.loc[contract_churn['ChurnRate'].idxmax(), 'Contract']
    h_contract_rate = contract_churn['ChurnRate'].max()
    l_contract      = contract_churn.loc[contract_churn['ChurnRate'].idxmin(), 'Contract']
    l_contract_rate = contract_churn['ChurnRate'].min()

    # Tenure
    df_lt12            = df[df['tenure'] <= 12]
    df_gt24            = df[df['tenure'] >  24]
    short_tenure_churn = df_lt12['Churn'].mean() * 100
    long_tenure_churn  = df_gt24['Churn'].mean() * 100

    # Internet service
    inet_churn  = churn_rate_by(df, 'InternetService')
    h_inet      = inet_churn.loc[inet_churn['ChurnRate'].idxmax(), 'InternetService']
    h_inet_rate = inet_churn['ChurnRate'].max()
    l_inet      = inet_churn.loc[inet_churn['ChurnRate'].idxmin(), 'InternetService']
    l_inet_rate = inet_churn['ChurnRate'].min()

    # Demographics
    senior_churn     = df[df['SeniorCitizen']==1]['Churn'].mean() * 100
    non_senior_churn = df[df['SeniorCitizen']==0]['Churn'].mean() * 100

    # Support services — actual rates, no vague wording
    ts_churn    = churn_rate_by(df, 'TechSupport')
    ts_no_rate  = float(ts_churn.loc[ts_churn['TechSupport']=='No',  'ChurnRate'].values[0]) \
                  if 'No'  in ts_churn['TechSupport'].values else float('nan')
    ts_yes_rate = float(ts_churn.loc[ts_churn['TechSupport']=='Yes', 'ChurnRate'].values[0]) \
                  if 'Yes' in ts_churn['TechSupport'].values else float('nan')
    sec_churn    = churn_rate_by(df, 'OnlineSecurity')
    sec_no_rate  = float(sec_churn.loc[sec_churn['OnlineSecurity']=='No',  'ChurnRate'].values[0]) \
                   if 'No'  in sec_churn['OnlineSecurity'].values else float('nan')
    sec_yes_rate = float(sec_churn.loc[sec_churn['OnlineSecurity']=='Yes', 'ChurnRate'].values[0]) \
                   if 'Yes' in sec_churn['OnlineSecurity'].values else float('nan')

    # Top feature from the best model's importance table
    top_feature = fi_df.iloc[0]['OriginalFeature']

    # Model-predicted risk (not confirmed churn outcomes)
    high_risk       = pred_df[pred_df['RiskCategory']=='High Risk']
    revenue_at_risk = high_risk['MonthlyCharges'].sum()
    high_risk_count = len(high_risk)

    # Payment method
    pay_churn  = churn_rate_by(df, 'PaymentMethod')
    h_pay      = pay_churn.loc[pay_churn['ChurnRate'].idxmax(), 'PaymentMethod']
    h_pay_rate = pay_churn['ChurnRate'].max()
    l_pay      = pay_churn.loc[pay_churn['ChurnRate'].idxmin(), 'PaymentMethod']
    l_pay_rate = pay_churn['ChurnRate'].min()

    loyal_count = len(df[(df['tenure'] > 24) & (df['Churn']==0)])

    return {
        'key_findings': [
            (f"Customers on {h_contract} contracts have the highest observed churn rate at "
             f"{h_contract_rate:.1f}% vs {l_contract_rate:.1f}% for {l_contract} contracts "
             f"(from {len(df):,} cleaned records)."),
            (f"Of {len(df_lt12):,} customers with tenure ≤ 12 months, {short_tenure_churn:.1f}% "
             f"have churned vs {long_tenure_churn:.1f}% for customers with tenure > 24 months."),
            (f"{h_inet} internet service customers show the highest observed churn rate at "
             f"{h_inet_rate:.1f}% vs {l_inet_rate:.1f}% for {l_inet} service. "
             f"Observed association — cause not established by this analysis."),
            (f"Senior citizens churn at {senior_churn:.1f}% vs {non_senior_churn:.1f}% for "
             f"non-seniors ({abs(senior_churn - non_senior_churn):.1f} pp difference)."),
            (f"'{top_feature}' has the highest aggregated importance score in the {model_name} "
             f"model (association measure, not causal evidence)."),
        ],
        'risks': [
            (f"{high_risk_count:,} customers ({high_risk_count/len(df)*100:.1f}%) are classified "
             f"as High Risk by the ML model (predicted churn probability > {MEDIUM_RISK_THRESHOLD:.0%})."),
            (f"Combined monthly charges for model-predicted High-Risk customers: "
             f"${revenue_at_risk:,.0f}/mo — potential exposure if predictions are correct, "
             f"not confirmed lost revenue."),
            (f"'{h_pay}' users have the highest observed churn rate at {h_pay_rate:.1f}% "
             f"vs {l_pay_rate:.1f}% for '{l_pay}' users "
             f"({round(h_pay_rate - l_pay_rate, 1):.1f} pp gap)."),
        ],
        'opportunities': [
            (f"{loyal_count:,} customers have tenure > 24 months and have not churned "
             f"({loyal_count/len(df)*100:.1f}% of all customers)."),
            (f"Customers without TechSupport churn at {ts_no_rate:.1f}% vs {ts_yes_rate:.1f}% "
             f"with it; without OnlineSecurity at {sec_no_rate:.1f}% vs {sec_yes_rate:.1f}%. "
             f"Correlational finding — controlled experiment needed to confirm causation."),
            (f"Observed churn gap: {h_contract} ({h_contract_rate:.1f}%) vs "
             f"{l_contract} ({l_contract_rate:.1f}%) = "
             f"{round(h_contract_rate - l_contract_rate, 1):.1f} pp — motivates a "
             f"contract-upgrade experiment."),
        ],
        'actions': [
            (f"Use the model's High-Risk flag (probability > {MEDIUM_RISK_THRESHOLD:.0%}) as a "
             f"prioritisation tool for the {high_risk_count:,} customers most worth investigating "
             f"for retention outreach. Validate through a controlled experiment before scaling."),
            (f"Evaluate targeted TechSupport and OnlineSecurity offers for customers who lack "
             f"these services and measure the retention impact experimentally. "
             f"Observed: TechSupport {ts_no_rate:.1f}% (no) vs {ts_yes_rate:.1f}% (yes); "
             f"OnlineSecurity {sec_no_rate:.1f}% (no) vs {sec_yes_rate:.1f}% (yes). "
             f"Correlational — not proven causal."),
            (f"Test contract-upgrade incentives among {h_contract} customers and measure "
             f"whether the intervention reduces churn. Observed gap: "
             f"{round(h_contract_rate - l_contract_rate, 1):.1f} pp."),
            (f"Investigate whether the higher churn rate among '{h_pay}' users ({h_pay_rate:.1f}%) "
             f"reflects a billing experience problem or a customer-segment effect. "
             f"Design an experiment before concluding that a payment-method change reduces churn."),
        ],
    }


# Use the best model's feature importance — not always rf_fi
fi_best  = rf_fi if best_name == 'Random Forest' else lr_fi
insights = generate_business_insights(df, kpis, fi_best, pred_df, best_name)

print('=== EXECUTIVE INSIGHTS ===')
for section, items in insights.items():
    print(f'\n--- {section.upper().replace("_", " ")} ---')
    for i, item in enumerate(items, 1):
        print(f'  {i}. {item}')

---
## 12. Save Model & Summary

In [ ]:
# Save the best model pipeline to disk
MODEL_PATH = 'churn_model_pipeline.joblib'

joblib.dump({
    'pipeline'  : best_pipe,
    'results'   : {'Logistic Regression': lr_metrics, 'Random Forest': rf_metrics},
    'best_name' : best_name,
    'cat_cols'  : cat_cols,
    'num_cols'  : num_cols,
    'X_test'    : X_test,
    'y_test'    : y_test,
}, MODEL_PATH)

print(f'Model pipeline saved to: {MODEL_PATH}')

In [ ]:
# --- Final Summary ---
print('=' * 60)
print('  TELECOM CHURN PREDICTION PROJECT — FINAL SUMMARY')
print('=' * 60)
print(f"  Dataset        : {kpis['Total Customers']:,} customers, {df.shape[1]-1} features")
print(f"  Churn Rate     : {kpis['Churn Rate (%)']:.2f}%")
print(f"  Retention Rate : {kpis['Retention Rate (%)']:.2f}%")
print(f"  Monthly Revenue: ${kpis['Total Monthly Revenue']:,.0f}")
print()
print(f"  Best Model     : {best_name}")
bm = rf_metrics if best_name == 'Random Forest' else lr_metrics
print(f"  Accuracy       : {bm['Accuracy']:.4f}")
print(f"  ROC-AUC        : {bm['ROC-AUC']:.4f}")
print(f"  Recall (Churn) : {bm['Recall']:.4f}")
print()
high_risk = pred_df[pred_df['RiskCategory']=='High Risk']
print(f"  High-Risk Customers : {len(high_risk):,}")
print(f"  Revenue at Risk     : ${high_risk['MonthlyCharges'].sum():,.0f}/mo")
print(f"  Top Churn Driver    : {fi_best.iloc[0]['OriginalFeature']} (from {best_name} model)")
print('=' * 60)
print()
print('To run the Streamlit dashboard:')
print('  streamlit run ShivamSingh_TelecomCustomerChurn.py')

---
## Summary & Conclusions

### What We Did
1. **Loaded and cleaned** the Telco Customer Churn dataset. The final analytical dataset contains `len(df)` customers after removing rows with blank `TotalCharges` and any exact duplicate rows (exact counts are printed by `clean_data()` above — no values are hard-coded here).
2. **Performed EDA** across 12 dimensions: contract, tenure, internet service, payment method, gender, senior status, monthly charges, tech support, online security, paperless billing, and more.
3. **Calculated business KPIs** including churn rate, retention rate, monthly revenue, and average tenure.
4. **Trained two ML models** — Logistic Regression and Random Forest — using a scikit-learn Pipeline with StandardScaler + OneHotEncoder preprocessing.
5. **Evaluated models** using Accuracy, Precision, Recall, F1 Score, and ROC-AUC. Selected the best model by ROC-AUC.
6. **Assigned risk categories** (Low / Medium / High) to every customer based on predicted churn probability.
7. **Identified top churn drivers** using feature importance.
8. **Generated executive insights** and recommended actions — all grounded in data.

### Key Takeaways
- **Contract type** is a dominant churn factor — month-to-month customers churn far more than annual/two-year customers.
- **Early tenure** (0–12 months) is the most vulnerable period — churn rate is highest in this window.
- **Fiber optic** internet service customers show a higher observed churn rate than DSL customers. The dataset alone does not establish why this difference occurs.
- Customers **without TechSupport or OnlineSecurity** show higher observed churn rates than customers with these services (exact rates computed and printed above).
- **Electronic check** payment method is associated with higher churn — this may reflect billing dissatisfaction.

### Business Impact
- Predicting churn *before* it happens enables the business to deploy **targeted retention actions**.
- A retention improvement scenario could be evaluated using the model-predicted high-risk population and their monthly charges; the actual business impact would depend on intervention effectiveness and retention-programme costs.

### Disclaimer
Feature importance indicates **association/correlation**, not causation. Retention strategies should be validated with controlled experiments (A/B tests) before full deployment.

---
*IBM SkillsBuild Data Analytics with AI Academic Internship | Shivam Singh*